# Final Flickr8k Text-to-Image Retrieval Project
### WiSE-FT CLIP Top-30 → BLIP Base / BLIP Large / BLIP-2 ITM → Learned Listwise Stacker

**Complete Kaggle notebook: data → CLIP processing → WiSE-FT → frozen rerankers → group-disjoint stacking → validation → optional test → exported artifacts.**

This notebook preserves the best *pilot* configuration from our sequence of experiments and removes components that did not help: **Qwen MLLM gate, BLIP LoRA/listwise retraining, SigLIP union, and QB-Norm**. The previous **700-query validation pilot** gave **83.14% T2I Recall@1** for the learned stacker (BLIP Large 82.14%, BLIP-2 ITM 82.57%). **These are previous results, not guaranteed notebook outputs.** Full-validation and final-test performance must be measured here.

> **Scientific limitations**
> - This is a custom **image-disjoint Flickr8k split**, not the official benchmark split. Use official splits before comparing to literature.
> - `Salesforce/blip-itm-base-coco` and `...large-coco` are COCO fine-tuned; full pretraining overlap with Flickr8k/Flickr30k has **not been ruled out**. Report this caveat.
> - The original WiSE-FT coefficient `alpha=0.20` was already selected using validation examples in earlier experiments, and the test split has been viewed repeatedly. New results from the same split are **not an untouched independent replication**.
> - The **stacker is fitted using validation image groups** and evaluated with image-disjoint out-of-fold predictions. The **train split** is used for CLIP projection tuning only; **test labels are never used for model selection**.
> - Retrieval scores are for **Text → Image**. Standard T2I evaluation uses one annotated target image per caption, although other images may be semantically valid.

**Runtime:** Internet ON, Kaggle GPU T4 (or better). With all three rerankers on the full validation/test sets this can take **tens of minutes** and requires several GB of model downloads. Disk caches let you resume successfully finished stages.

**Reference checkpoints**: [CLIP ViT-B/32](https://huggingface.co/openai/clip-vit-base-patch32) · [BLIP Base ITM](https://huggingface.co/Salesforce/blip-itm-base-coco) · [BLIP Large ITM](https://huggingface.co/Salesforce/blip-itm-large-coco) · [BLIP-2 ITM ViT-G](https://huggingface.co/Salesforce/blip2-itm-vit-g)

## 1 · Imports, configuration and reproducibility

In [ ]:
import gc, hashlib, json, random, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, GroupKFold, GroupShuffleSplit
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoProcessor, CLIPModel,
    BlipProcessor, BlipForImageTextRetrieval,
    Blip2ForImageTextRetrieval,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Enable a GPU in Kaggle notebook settings.'

DATA_ROOT = Path('/kaggle/input/datasets/adityajn105/flickr8k')
IMAGE_DIR = DATA_ROOT / 'Images'
CAPTION_FILE = DATA_ROOT / 'captions.txt'
OUT = Path('/kaggle/working/final_wise_blip_stacker')
OUT.mkdir(parents=True, exist_ok=True)

CLIP_ID = 'openai/clip-vit-base-patch32'
RERANKERS = {
    'base': 'Salesforce/blip-itm-base-coco',
    'large': 'Salesforce/blip-itm-large-coco',
    'blip2': 'Salesforce/blip2-itm-vit-g',
}
EXISTING_WISE = Path('/kaggle/working/clip_wiseft/wiseft_best.pt')
# If EXISTING_WISE is missing, train projection layers here and blend them.
WISE_ALPHA = 0.20             # fixed from prior experiment, never tuned on test
TOP_K = 30
PILOT_IMAGE_COUNT = None    # None: all 1,011 validation images (~5,055 queries)
RUN_FINAL_TEST = True       # Runs after out-of-fold validation and final stacker fitting
CV_FOLDS = 4
N_BOOTSTRAP = 2000
CLIP_BATCH = 16
BLIP_VISION_BATCH = 8
BLIP_PAIR_BATCH = 16
BLIP2_VISION_BATCH = 4
BLIP2_PAIR_BATCH = 8
TRAIN_BATCH = 8
TRAIN_EPOCHS = 6
TRAIN_PATIENCE = 3
TRAIN_LR = 1e-4

print('GPU:', torch.cuda.get_device_name(0))
print('Dataset:', DATA_ROOT)
print('Output:', OUT)
print('Full validation:', PILOT_IMAGE_COUNT is None)
print('Final test enabled:', RUN_FINAL_TEST)
print('All rerankers:', list(RERANKERS.values()))

## 2 · Inspect, validate and split Flickr8k (image-disjoint)

In [ ]:
assert IMAGE_DIR.is_dir(), f'Image directory missing: {IMAGE_DIR}'
assert CAPTION_FILE.exists(), f'Captions file missing: {CAPTION_FILE}'

df = pd.read_csv(CAPTION_FILE)
df.columns = df.columns.str.strip().str.lower()
assert {'image', 'caption'}.issubset(df.columns), df.columns.tolist()
raw_rows = len(df)
df = df.dropna(subset=['image','caption']).copy()
df['image'] = df['image'].astype(str).str.strip()
df['caption'] = df['caption'].astype(str).str.strip()
df = df[(df.image != '') & (df.caption != '')]
df = df.drop_duplicates(['image','caption']).copy()

allowed = {'.jpg', '.jpeg', '.png'}
files = {p.name: p for p in IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() in allowed}
missing_files = sorted(set(df.image) - set(files))
df = df[df.image.isin(files)].copy()

# Verify image integrity once (does not decode/preprocess images for training).
valid, corrupt = set(), []
for name in tqdm(sorted(df.image.unique()), desc='Verify images'):
    try:
        with Image.open(files[name]) as im:
            im.verify()
        valid.add(name)
    except Exception as exc:
        corrupt.append((name, str(exc)))
df = df[df.image.isin(valid)].reset_index(drop=True)
assert not df.empty, 'No valid caption pairs found.'

caption_map = df.groupby('image', sort=False)['caption'].apply(list).to_dict()
all_image_names = sorted(caption_map)
train_images, temp_images = train_test_split(all_image_names, test_size=0.25, random_state=SEED)
val_images, test_images = train_test_split(temp_images, test_size=0.5, random_state=SEED)
train_images, val_images, test_images = list(train_images), list(val_images), list(test_images)
assert set(train_images).isdisjoint(val_images)
assert set(train_images).isdisjoint(test_images)
assert set(val_images).isdisjoint(test_images)
assert len(train_images)+len(val_images)+len(test_images)==len(all_image_names)

split = {**{i:'train' for i in train_images},
         **{i:'validation' for i in val_images},
         **{i:'test' for i in test_images}}
df['split'] = df.image.map(split)
df.to_csv(OUT/'cleaned_splits.csv',index=False)
print('Raw caption rows:', raw_rows, '| cleaned rows:', len(df))
print('Missing images:',len(missing_files),'| corrupt images:',len(corrupt))
for name, ids in [('Train',train_images),('Validation',val_images),('Test',test_images)]:
    print(f'{name:<12}: {len(ids):>5} images; {sum(len(caption_map[n]) for n in ids):>6} captions')
print('Captions/image:', df.groupby('image').size().value_counts().sort_index().to_dict())

## 3 · CLIP Dataset/DataLoader and official preprocessing

In [ ]:
clip_processor = AutoProcessor.from_pretrained(CLIP_ID)

class FlickrDataset(Dataset):
    def __init__(self, names):
        self.names = list(names)
    def __len__(self):
        return len(self.names)
    def __getitem__(self, i):
        name = self.names[i]
        with Image.open(IMAGE_DIR / name) as im:
            image = im.convert('RGB').copy()
        return {'image': image, 'name': name, 'captions': caption_map[name]}

def clip_collate(batch):
    images = [x['image'] for x in batch]
    names = [x['name'] for x in batch]
    captions, owners = [], []
    for j,item in enumerate(batch):
        captions.extend(item['captions'])
        owners.extend([j]*len(item['captions']))
    images_enc = clip_processor.image_processor(images=images,return_tensors='pt')
    texts_enc = clip_processor.tokenizer(
        captions,padding='max_length',truncation=True,max_length=77,return_tensors='pt')
    return {'pixel_values':images_enc['pixel_values'],
            'input_ids':texts_enc['input_ids'],
            'attention_mask':texts_enc['attention_mask'],
            'names':names, 'captions':captions,
            'caption_to_image':torch.tensor(owners,dtype=torch.long)}

def make_loader(names, shuffle=False):
    return DataLoader(FlickrDataset(names),batch_size=CLIP_BATCH,shuffle=shuffle,
                      num_workers=0,collate_fn=clip_collate)

train_loader=make_loader(train_images)
val_loader=make_loader(val_images)
test_loader=make_loader(test_images) if RUN_FINAL_TEST else None

b=next(iter(val_loader))
assert b['pixel_values'].shape[1:] == (3,224,224)
assert b['input_ids'].shape[1] == 77
print('Image batch:',tuple(b['pixel_values'].shape))
print('Caption batch:',tuple(b['input_ids'].shape))
print('Mapping:',tuple(b['caption_to_image'].shape))

## 4 · Cache frozen CLIP encoder features (reused throughout)

In [ ]:
def sha(*values):
    h=hashlib.sha256()
    for value in values:
        if isinstance(value,np.ndarray):h.update(np.ascontiguousarray(value).tobytes())
        elif isinstance(value,(list,tuple)):h.update('\n'.join(map(str,value)).encode())
        else:h.update(str(value).encode())
    return h.hexdigest()[:16]

def read_hidden_cache(filename, expect_names):
    if not filename.exists(): return None
    obj=torch.load(filename,map_location='cpu',weights_only=True)
    if obj.get('names') != list(expect_names):return None
    if len(obj['captions'])!=len(obj['mapping']):return None
    return obj

@torch.inference_mode()
def extract_hidden(loader, clip, title):
    v,t,names,captions,mapping=[],[],[],[],[]
    clip.eval()
    for batch in tqdm(loader,desc=f'CLIP encoder: {title}'):
        offset=len(names)
        pixel=batch['pixel_values'].to(DEVICE)
        ids=batch['input_ids'].to(DEVICE)
        att=batch['attention_mask'].to(DEVICE)
        with torch.autocast('cuda',dtype=torch.float16):
            vh=clip.vision_model(pixel_values=pixel).pooler_output
            th=clip.text_model(input_ids=ids,attention_mask=att).pooler_output
        v.append(vh.float().cpu());t.append(th.float().cpu())
        names.extend(batch['names']);captions.extend(batch['captions'])
        mapping.extend((batch['caption_to_image']+offset).tolist())
    result={'v':torch.cat(v), 't':torch.cat(t),
            'names':names,'captions':captions,
            'mapping':torch.tensor(mapping,dtype=torch.long)}
    assert len(result['names']) == len(set(result['names']))
    return result

wanted={'train':(train_images,train_loader),'val':(val_images,val_loader)}
if RUN_FINAL_TEST: wanted['test']=(test_images,test_loader)
cache_files={k:OUT/f'clip_hidden_{k}_{sha(names,CLIP_ID,"split42_v1")}.pt'
             for k,(names,_) in wanted.items()}
hidden={k:read_hidden_cache(cache_files[k],names) for k,(names,_) in wanted.items()}
missing=[k for k,v in hidden.items() if v is None]

clip=CLIPModel.from_pretrained(CLIP_ID).to(DEVICE).eval()
clip.requires_grad_(False)
PRE_V=clip.visual_projection.weight.detach().float().cpu().clone()
PRE_T=clip.text_projection.weight.detach().float().cpu().clone()
LOGIT_SCALE=float(clip.logit_scale.exp().detach().cpu())

for key in missing:
    hidden[key]=extract_hidden(wanted[key][1],clip,key)
    torch.save(hidden[key],cache_files[key])
    print('Cached:',cache_files[key].name)

train_cache=hidden['train'];val_cache=hidden['val']
test_cache=hidden.get('test')
assert set(train_cache['names']).isdisjoint(val_cache['names'])
if test_cache is not None:
    assert set(test_cache['names']).isdisjoint(val_cache['names'])
    assert set(test_cache['names']).isdisjoint(train_cache['names'])

print('Frozen CLIP hidden shapes (train):',tuple(train_cache['v'].shape),tuple(train_cache['t'].shape))
print('Frozen CLIP hidden shapes (val):',tuple(val_cache['v'].shape),tuple(val_cache['t'].shape))
del clip
gc.collect()
torch.cuda.empty_cache()

## 5 · Projection-only CLIP fine-tuning and fixed WiSE-FT interpolation

If `/kaggle/working/clip_wiseft/wiseft_best.pt` exists, reuse that **exact previously evaluated projection checkpoint** (avoids retraining and preserves comparability). Otherwise, train only the visual/text linear projections from the original CLIP checkpoint, select the best epoch by **validation Mean Recall@1**, and form `W* = 0.8 W_pretrained + 0.2 W_finetuned`. The **0.20 coefficient remains fixed** in this notebook.

> Feature caching differs slightly from an online training pass, but both CLIP encoders are fully frozen. A fresh run can therefore produce different absolute metrics than the original project run.

In [ ]:
@torch.inference_mode()
def project_and_retrieve(cache, wv, wt, k=30):
    v=F.normalize(F.linear(cache['v'].float().to(DEVICE),wv),dim=-1)
    t=F.normalize(F.linear(cache['t'].float().to(DEVICE),wt),dim=-1)
    k=min(k,len(v))
    ix, sc=[],[]
    for j in range(0,len(t),256):
        vals,inds=(t[j:j+256]@v.T).topk(k,dim=1)
        ix.append(inds.cpu().numpy());sc.append(vals.cpu().numpy())
    return np.concatenate(ix),np.concatenate(sc)

def retrieval_metrics(candidate_ids, ground_truth, ks=(1,5,10)):
    y=np.asarray(ground_truth,dtype=np.int64)
    assert len(candidate_ids)==len(y)
    return {f'R@{k}':100*float(np.mean(np.any(candidate_ids[:,:min(k,candidate_ids.shape[1])]
                                                      ==y[:,None],axis=1))) for k in ks}

def symmetric_multi_positive(logits, owners):
    n_img,n_cap=logits.shape
    im=torch.arange(n_img,device=logits.device)
    targets=(im[:,None]==owners[None,:]).float()
    targets=targets/targets.sum(dim=1,keepdim=True).clamp_min(1)
    i2t=-(targets*F.log_softmax(logits.float(),dim=1)).sum(dim=1).mean()
    t2i=F.cross_entropy(logits.float().T,owners)
    return (i2t+t2i)/2

class Projections(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual=nn.Linear(PRE_V.shape[1],PRE_V.shape[0],bias=False)
        self.text=nn.Linear(PRE_T.shape[1],PRE_T.shape[0],bias=False)
        with torch.no_grad():
            self.visual.weight.copy_(PRE_V)
            self.text.weight.copy_(PRE_T)
    def forward(self, v, t):
        return (F.normalize(self.visual(v),dim=-1),
                F.normalize(self.text(t),dim=-1))

train_map=train_cache['mapping'].numpy()
train_cap_group=[np.flatnonzero(train_map==i) for i in range(len(train_cache['names']))]

def train_projection_if_needed():
    if EXISTING_WISE.exists():
        ck=torch.load(EXISTING_WISE,map_location='cpu',weights_only=True)
        assert abs(float(ck['alpha'])-WISE_ALPHA)<1e-6
        print('Reusing previous WiSE-FT:',EXISTING_WISE)
        return ck['visual_projection'].float(),ck['text_projection'].float(),'existing_checkpoint'

    proj=Projections().to(DEVICE)
    optim=torch.optim.AdamW(proj.parameters(),lr=TRAIN_LR,weight_decay=0.01)
    best_score=-float('inf');best_w=None;wait=0
    rng=np.random.default_rng(SEED)
    print('Existing WiSE checkpoint missing. Training projection layers now.')
    for epoch in range(1,TRAIN_EPOCHS+1):
        proj.train();order=rng.permutation(len(train_cache['names']))
        total_loss=0.;count=0
        for start in tqdm(range(0,len(order),TRAIN_BATCH),desc=f'Projection epoch {epoch}',leave=False):
            ids=order[start:start+TRAIN_BATCH]
            cap_ids=np.concatenate([train_cap_group[int(i)] for i in ids])
            local_owner=np.concatenate([np.full(len(train_cap_group[int(i)]),j,dtype=np.int64)
                                        for j,i in enumerate(ids)])
            v=train_cache['v'][ids].to(DEVICE)
            t=train_cache['t'][cap_ids].to(DEVICE)
            owners=torch.tensor(local_owner,device=DEVICE)
            optim.zero_grad(set_to_none=True)
            ev,et=proj(v,t)
            loss=symmetric_multi_positive(LOGIT_SCALE*(ev@et.T),owners)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(proj.parameters(),1.0)
            optim.step()
            total_loss+=float(loss.item())*len(ids);count+=len(ids)
        proj.eval()
        wv=proj.visual.weight.detach().float()
        wt=proj.text.weight.detach().float()
        cand,_=project_and_retrieve(val_cache,wv,wt,k=1)
        val_gt=val_cache['mapping'].numpy()
        # T2I: query captions against validation images
        val_t2i=float(np.mean(cand[:,0]==val_gt))*100
        # I2T: any caption of the image ranked first
        with torch.inference_mode():
            ve=F.normalize(F.linear(val_cache['v'].to(DEVICE),wv),dim=-1)
            te=F.normalize(F.linear(val_cache['t'].to(DEVICE),wt),dim=-1)
            top_caption=(ve@te.T).argmax(dim=1).cpu().numpy()
        val_i2t=float(np.mean(val_gt[top_caption]==np.arange(len(ve))))*100
        mean_r1=(val_i2t+val_t2i)/2
        print(f'Epoch {epoch}: loss={total_loss/count:.4f}; val I2T={val_i2t:.2f}%; '
              f'val T2I={val_t2i:.2f}%; mean={mean_r1:.2f}%')
        if mean_r1>best_score+1e-7:
            best_score=mean_r1;wait=0
            best_w=(wv.detach().cpu().clone(),wt.detach().cpu().clone())
        else:
            wait+=1
            if wait>=TRAIN_PATIENCE:
                print('Early stopped.');break
    assert best_w is not None
    del proj,optim
    gc.collect();torch.cuda.empty_cache()
    wv=(1-WISE_ALPHA)*PRE_V+WISE_ALPHA*best_w[0]
    wt=(1-WISE_ALPHA)*PRE_T+WISE_ALPHA*best_w[1]
    return wv,wt,'trained_in_this_notebook'

WV,WT,wise_source=train_projection_if_needed()
WV=WV.to(DEVICE).float();WT=WT.to(DEVICE).float()
assert WV.shape==PRE_V.shape and WT.shape==PRE_T.shape
wise_save=OUT/'wiseft_projections.pt'
torch.save({'model_id':CLIP_ID,'alpha':WISE_ALPHA,
            'visual_projection':WV.cpu(),'text_projection':WT.cpu(),
            'source':wise_source},wise_save)
print('WiSE-FT projection checkpoint:',wise_save,'| source:',wise_source)

## 6 · WiSE-FT Top-30 candidate retrieval (full image gallery)

In [ ]:
def make_retrieval(cache, split_name, select_image_count=None):
    all_names=cache['names'];all_caps=cache['captions']
    gt=cache['mapping'].numpy()
    if select_image_count is None:
        qids=np.arange(len(gt))
    else:
        rng=np.random.default_rng(SEED)
        selected=rng.choice(len(all_names),min(select_image_count,len(all_names)),replace=False)
        qids=np.flatnonzero(np.isin(gt,selected))
    # Keep FULL split gallery even in a pilot.
    subset_cache={'v':cache['v'],'t':cache['t'][qids]}
    candidates,clip_scores=project_and_retrieve(subset_cache,WV,WT,k=TOP_K)
    assert candidates.shape==(len(qids),TOP_K)
    result={'split':split_name,'names':all_names,
            'texts':[all_caps[i] for i in qids],
            'query_ids':qids,
            'gt':gt[qids],
            'candidates':candidates,
            'clip_scores':clip_scores}
    print(f'{split_name}: {len(result["texts"])} queries, {len(all_names)} gallery images, '
          f'Top-{TOP_K} coverage={100*np.mean(np.any(candidates==gt[qids,None],axis=1)):.2f}%')
    return result

val_data=make_retrieval(val_cache,'validation',PILOT_IMAGE_COUNT)
print('WiSE retrieval:',retrieval_metrics(val_data['candidates'],val_data['gt']))

## 7 · Frozen reranker scores: BLIP Base, BLIP Large, BLIP-2 ITM

Each model is **loaded and released sequentially** to fit a T4. Vision tokens are cached per unique gallery image; each `(caption, candidate image)` pair is then scored using the model's **ITM matching vs non-matching logit difference**. A model's score is never treated as a calibrated probability. All three scorers are **frozen**, and no BLIP fine-tuning is included.

The BLIP-2 model must be `Salesforce/blip2-itm-vit-g` (actual ITM checkpoint), not a generative BLIP-2 OPT model. Restarting the notebook reuses successfully completed score caches; if changing model IDs or WiSE weights, caches automatically receive new fingerprints.

In [ ]:
def query_score_features(scores):
    scores=np.asarray(scores,dtype=np.float32)
    # Per-query normalization is safe: it never uses ground-truth labels.
    z=(scores-scores.mean(axis=1,keepdims=True))/(scores.std(axis=1,keepdims=True)+1e-6)
    order=np.argsort(-scores,axis=1,kind='stable')
    ranks=np.empty_like(order)
    ranks[np.arange(len(order))[:,None],order]=np.arange(scores.shape[1])[None,:]
    rank01=1-ranks.astype(np.float32)/max(1,scores.shape[1]-1)
    margin=z-z.max(axis=1,keepdims=True)
    return z.astype('float32'),rank01.astype('float32'),margin.astype('float32')

def load_images(names_list):
    ans=[]
    for name in names_list:
        with Image.open(IMAGE_DIR/name) as img:
            ans.append(img.convert('RGB').copy())
    return ans

def score_key(data,model_id):
    return sha(data['names'],data['texts'],data['candidates'],
               model_id,WV.cpu().numpy(),WT.cpu().numpy(),'itm-v2')

def get_or_compute_vision(model,processor,data,used,model_id,kind):
    names=[data['names'][i] for i in used]
    key=sha(names,model_id,'vision-v2')
    dst=OUT/f'{data["split"]}_{kind}_vision_{key}.npy'
    if dst.exists():
        arr=np.load(dst,mmap_mode='r')
        if len(arr)==len(used) and arr.dtype==np.float16:
            print('Reuse vision tokens:',dst.name,arr.shape)
            return arr
        del arr
        dst.unlink()
    B=BLIP2_VISION_BATCH if kind=='blip2' else BLIP_VISION_BATCH
    mm=None
    for i in tqdm(range(0,len(used),B),desc=f'{kind} vision {data["split"]}'):
        batch_names=names[i:i+B]
        pictures=load_images(batch_names)
        pix=processor.image_processor(images=pictures,return_tensors='pt')['pixel_values'].to(DEVICE)
        with torch.inference_mode(),torch.autocast('cuda',dtype=torch.float16):
            tokens=model.vision_model(pixel_values=pix).last_hidden_state.float().cpu().numpy()
        if mm is None:
            shape=(len(used),tokens.shape[1],tokens.shape[2])
            required_bytes=int(np.prod(shape))*2
            if shutil.disk_usage(OUT).free < required_bytes + 300*1024*1024:
                raise RuntimeError('Insufficient Kaggle working disk for vision cache')
            mm=np.lib.format.open_memmap(dst,mode='w+',dtype=np.float16,shape=shape)
        mm[i:i+len(batch_names)]=tokens.astype(np.float16)
        del pix,pictures,tokens
    assert mm is not None
    mm.flush()
    del mm
    return np.load(dst,mmap_mode='r')

@torch.inference_mode()
def score_blip(data,model_id,kind):
    key=score_key(data,model_id)
    file=OUT/f'{data["split"]}_{kind}_scores_{key}.npy'
    n,k=data['candidates'].shape
    if file.exists():
        ans=np.load(file)
        if ans.shape==(n,k):
            print('Reuse scores:',file.name)
            return ans
    print('Loading:',model_id)
    processor=BlipProcessor.from_pretrained(model_id)
    model=BlipForImageTextRetrieval.from_pretrained(model_id,torch_dtype=torch.float16).to(DEVICE).eval()
    model.requires_grad_(False)
    used=np.unique(data['candidates'].reshape(-1))
    lookup=np.full(len(data['names']),-1,dtype=np.int32)
    lookup[used]=np.arange(len(used))
    vision=get_or_compute_vision(model,processor,data,used,model_id,kind)
    tokenized=processor.tokenizer(data['texts'],padding='max_length',truncation=True,max_length=35,return_tensors='pt')
    pairs_image=lookup[data['candidates'].reshape(-1)]
    pairs_query=np.repeat(np.arange(n,dtype=np.int64),k)
    out=np.empty(n*k,dtype=np.float32)
    for start in tqdm(range(0,n*k,BLIP_PAIR_BATCH),desc=f'{kind} score pairs'):
        end=min(start+BLIP_PAIR_BATCH,n*k)
        q=torch.as_tensor(pairs_query[start:end],dtype=torch.long)
        v=torch.from_numpy(np.array(vision[pairs_image[start:end]],copy=True)).to(DEVICE)
        ids=tokenized['input_ids'][q].to(DEVICE)
        mask=tokenized['attention_mask'][q].to(DEVICE)
        with torch.autocast('cuda',dtype=torch.float16):
            h=model.text_encoder(input_ids=ids,attention_mask=mask,
                                 encoder_hidden_states=v,return_dict=True).last_hidden_state[:,0,:]
            logits=model.itm_head(h).float()
        out[start:end]=(logits[:,1]-logits[:,0]).cpu().numpy()
    result=out.reshape(n,k)
    np.save(file,result)
    del model,processor,vision,tokenized
    gc.collect();torch.cuda.empty_cache()
    return result

@torch.inference_mode()
def score_blip2(data,model_id,kind='blip2'):
    key=score_key(data,model_id)
    file=OUT/f'{data["split"]}_{kind}_scores_{key}.npy'
    n,k=data['candidates'].shape
    if file.exists():
        ans=np.load(file)
        if ans.shape==(n,k):
            print('Reuse scores:',file.name)
            return ans
    print('Loading:',model_id)
    processor=AutoProcessor.from_pretrained(model_id)
    model=Blip2ForImageTextRetrieval.from_pretrained(model_id,torch_dtype=torch.float16).to(DEVICE).eval()
    model.requires_grad_(False)
    used=np.unique(data['candidates'].reshape(-1))
    lookup=np.full(len(data['names']),-1,dtype=np.int32)
    lookup[used]=np.arange(len(used))
    vision=get_or_compute_vision(model,processor,data,used,model_id,kind)
    tok=processor.tokenizer(data['texts'],padding='max_length',max_length=35,truncation=True,return_tensors='pt')
    pairs_image=lookup[data['candidates'].reshape(-1)]
    pairs_query=np.repeat(np.arange(n,dtype=np.int64),k)
    out=np.empty(n*k,dtype=np.float32)
    for start in tqdm(range(0,n*k,BLIP2_PAIR_BATCH),desc=f'{kind} ITM pairs'):
        end=min(start+BLIP2_PAIR_BATCH,n*k)
        q=torch.as_tensor(pairs_query[start:end],dtype=torch.long)
        vh=torch.from_numpy(np.array(vision[pairs_image[start:end]],copy=True)).to(DEVICE)
        ids=tok['input_ids'][q].to(DEVICE)
        mask=tok['attention_mask'][q].to(DEVICE)
        query_tokens=model.query_tokens.expand(len(q),-1,-1)
        with torch.autocast('cuda',dtype=torch.float16):
            embedded=model.embeddings(input_ids=ids,query_embeds=query_tokens)
            qmask=torch.ones(query_tokens.shape[:2],dtype=torch.long,device=DEVICE)
            att=torch.cat([qmask,mask],dim=1)
            vmask=torch.ones(vh.shape[:2],dtype=torch.long,device=DEVICE)
            fused=model.qformer(query_embeds=embedded,query_length=query_tokens.shape[1],
                                attention_mask=att,encoder_hidden_states=vh,
                                encoder_attention_mask=vmask,return_dict=True)
            logits=model.itm_head(fused.last_hidden_state[:,:query_tokens.shape[1],:]).mean(dim=1).float()
        out[start:end]=(logits[:,1]-logits[:,0]).cpu().numpy()
    result=out.reshape(n,k)
    np.save(file,result)
    del model,processor,vision,tok
    gc.collect();torch.cuda.empty_cache()
    return result

def frozen_scores(data):
    scores={'wise':data['clip_scores'].astype(np.float32)}
    scores['base']=score_blip(data,RERANKERS['base'],'base')
    scores['large']=score_blip(data,RERANKERS['large'],'large')
    scores['blip2']=score_blip2(data,RERANKERS['blip2'],'blip2')
    for name,sc in scores.items():
        assert sc.shape==data['candidates'].shape,(name,sc.shape)
        assert np.isfinite(sc).all(),name
    return scores

val_scores=frozen_scores(val_data)
print('Cached all frozen validation reranker scores.')

## 8 · Per-candidate features (16-D) and image-disjoint labels

In [ ]:
def make_stacker_features(scores,texts):
    # A 16-D vector: for each of four scorers, z / rank / margin / z×length.
    length=np.asarray([len(s.split()) for s in texts],dtype=np.float32)
    normalized_length=np.clip((length-10.0)/10.0,-1.0,2.0)
    chunks,names=[],[]
    for source in ('base','large','wise','blip2'):
        z,r,margin=query_score_features(scores[source])
        for suffix,value in [('z',z),('rank',r),('margin',margin),
                             ('z_x_length',z*normalized_length[:,None])]:
            chunks.append(value[...,None])
            names.append(source+'_'+suffix)
    return np.concatenate(chunks,axis=2).astype(np.float32),names

val_X,feature_names=make_stacker_features(val_scores,val_data['texts'])
val_candidates=val_data['candidates']
val_gt=np.asarray(val_data['gt'],dtype=np.int64)
pos=(val_candidates==val_gt[:,None])
val_y=np.where(pos.any(axis=1),pos.argmax(axis=1),-1).astype(np.int64)
assert val_X.shape==(len(val_gt),TOP_K,16)
assert 0<=val_y.max()<TOP_K
print('Features:',val_X.shape,'| positive candidate coverage:',float((val_y>=0).mean()))
print('Feature names:',feature_names)

## 9 · Image-disjoint OOF learned listwise stacker

The low-capacity model has **only 16 trainable scalar weights** and minimizes softmax cross-entropy over the 30 candidates. **All captions of each validation image belong to the same fold.** Positives outside the retrieved candidate set are omitted **only from stacker training** but count as misses at evaluation.

A small **inner image-disjoint holdout** stops fitting in each outer fold. The outer holdout receives only predictions from weights trained without its image group. This is out-of-fold validation, not a wholly new held-out dataset.

In [ ]:
def train_stacker_inner(X,y,groups,seed=SEED,max_epochs=160,patience=20):
    positive=np.flatnonzero(y>=0)
    assert len(positive)>25
    inner=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=seed)
    a,b=next(inner.split(positive,groups=groups[positive]))
    tr=positive[a];va=positive[b]
    Xt=torch.from_numpy(X[tr]).to(DEVICE)
    yt=torch.from_numpy(y[tr]).to(DEVICE)
    Xv=torch.from_numpy(X[va]).to(DEVICE)
    yv=torch.from_numpy(y[va]).to(DEVICE)
    w=nn.Parameter(torch.zeros(X.shape[2],device=DEVICE))
    optim=torch.optim.AdamW([w],lr=0.05,weight_decay=0.03)
    best_loss=float('inf');best_weight=None;best_epoch=0;wait=0
    for epoch in range(1,max_epochs+1):
        optim.zero_grad(set_to_none=True)
        logits=torch.einsum('qkf,f->qk',Xt,w)
        loss=F.cross_entropy(logits,yt)+0.01*w.square().mean()
        loss.backward();optim.step()
        with torch.no_grad():
            monitor=F.cross_entropy(torch.einsum('qkf,f->qk',Xv,w),yv).item()
        if monitor < best_loss-1e-5:
            best_loss=monitor;best_weight=w.detach().cpu().numpy().copy()
            best_epoch=epoch;wait=0
        else:
            wait+=1
            if wait>=patience:break
    return best_weight,best_epoch

def fit_stacker_fixed_epochs(X,y,num_epochs):
    chosen=np.flatnonzero(y>=0)
    Xt=torch.from_numpy(X[chosen]).to(DEVICE)
    yt=torch.from_numpy(y[chosen]).to(DEVICE)
    w=nn.Parameter(torch.zeros(X.shape[-1],device=DEVICE))
    optim=torch.optim.AdamW([w],lr=0.05,weight_decay=0.03)
    for _ in range(num_epochs):
        optim.zero_grad(set_to_none=True)
        loss=F.cross_entropy(torch.einsum('qkf,f->qk',Xt,w),yt)+0.01*w.square().mean()
        loss.backward();optim.step()
    return w.detach().cpu().numpy().copy()

def rank_candidates(candidates,scores):
    order=np.argsort(-scores,axis=1,kind='stable')
    return np.take_along_axis(candidates,order,axis=1)

kfold=GroupKFold(n_splits=CV_FOLDS)
oof_logits=np.zeros(val_y.shape+(TOP_K,),dtype=np.float32)
cv_log=[]
cv_splits=list(kfold.split(val_X,groups=val_gt))
for fold,(tr,ev) in enumerate(cv_splits,1):
    assert set(val_gt[tr]).isdisjoint(set(val_gt[ev]))
    w,epoch=train_stacker_inner(val_X[tr],val_y[tr],val_gt[tr],seed=SEED+fold)
    oof_logits[ev]=val_X[ev]@w
    cv_log.append({'fold':fold,'train_queries':len(tr),'eval_queries':len(ev),'best_inner_epoch':epoch})
    print(f'Fold {fold}: train={len(tr)}, OOF={len(ev)}, inner best epoch={epoch}')
cv_table=pd.DataFrame(cv_log)
display(cv_table)
cv_table.to_csv(OUT/'stacker_oof_folds.csv',index=False)
np.save(OUT/'oof_stacker_scores.npy',oof_logits)

## 10 · Full-validation OOF baselines, confidence intervals, caption length and plots

In [ ]:
def calc_metrics(name,ranks,gt,texts):
    result={'Model':name,**retrieval_metrics(ranks,gt)}
    length=np.asarray([len(t.split()) for t in texts])
    for title,mask in [('short',length<=7),('medium',(length>7)&(length<14)),('long',length>=14)]:
        result[title+'_count']=int(mask.sum())
        result[title+'_R1']=float(100*np.mean(ranks[mask,0]==gt[mask])) if mask.any() else np.nan
    return result

# Cross-validated prior rank-fusion baselines (same folds as stacker).
base_rank=query_score_features(val_scores['base'])[1]
large_rank=query_score_features(val_scores['large'])[1]
length=np.asarray([len(t.split()) for t in val_data['texts']])
length_buckets=np.where(length<=7,'short',np.where(length>=14,'long','medium'))

def best_rank_fusion_weight(idx,bucket=None):
    selected=idx if bucket is None else idx[length_buckets[idx]==bucket]
    if len(selected)<10:return 0.5
    best=(-1.,0.5)
    for weight in np.linspace(0,1,11):
        scores=weight*base_rank[selected]+(1-weight)*large_rank[selected]
        pred=val_candidates[selected,np.argmax(scores,axis=1)]
        acc=float(np.mean(pred==val_gt[selected]))
        if acc>best[0]+1e-9:
            best=(acc,float(weight))
    return best[1]

prior_global=np.empty_like(oof_logits)
prior_length=np.empty_like(oof_logits)
for tr,ev in cv_splits:
    ww=best_rank_fusion_weight(tr)
    prior_global[ev]=ww*base_rank[ev]+(1-ww)*large_rank[ev]
    for key in ('short','medium','long'):
        ids=ev[length_buckets[ev]==key]
        if len(ids)==0:continue
        w=best_rank_fusion_weight(tr,key)
        prior_length[ids]=w*base_rank[ids]+(1-w)*large_rank[ids]

ranks={
    'WiSE-FT Top-30':val_candidates,
    'BLIP Base':rank_candidates(val_candidates,val_scores['base']),
    'BLIP Large':rank_candidates(val_candidates,val_scores['large']),
    'BLIP-2 ITM':rank_candidates(val_candidates,val_scores['blip2']),
    'Prior Global Fusion OOF':rank_candidates(val_candidates,prior_global),
    'Prior Length-aware Fusion OOF':rank_candidates(val_candidates,prior_length),
    'Learned Stacker OOF':rank_candidates(val_candidates,oof_logits),
}
results=pd.DataFrame([calc_metrics(n,v,val_gt,val_data['texts']) for n,v in ranks.items()]).set_index('Model')
print('=== FULL VALIDATION, IMAGE-DISJOINT OOF ===')
display(results.round(3))
results.to_csv(OUT/'full_validation_results.csv')

# Paired cluster bootstrap over the SOURCE IMAGE, not individual captions.
def paired_image_bootstrap(hits_a,hits_b,group_ids,n_boot=2000,seed=SEED):
    hits_a=np.asarray(hits_a,dtype=np.float32)
    hits_b=np.asarray(hits_b,dtype=np.float32)
    group_ids=np.asarray(group_ids,dtype=np.int64)
    ids,reverse=np.unique(group_ids,return_inverse=True)
    diff=hits_a-hits_b
    sums=np.bincount(reverse,weights=diff,minlength=len(ids))
    counts=np.bincount(reverse,minlength=len(ids))
    rng=np.random.default_rng(seed)
    boot=[]
    for _ in range(n_boot):
        resample=rng.integers(0,len(ids),len(ids))
        boot.append(100*sums[resample].sum()/counts[resample].sum())
    return (100*diff.mean(),*np.percentile(boot,[2.5,97.5]))

baseline_hit=ranks['BLIP Large'][:,0]==val_gt
stack_hit=ranks['Learned Stacker OOF'][:,0]==val_gt
records=[]
for title,mask in [('overall',np.ones(len(val_gt),dtype=bool)),
                   ('short',length<=7),('medium',(length>7)&(length<14)),('long',length>=14)]:
    d,lo,hi=paired_image_bootstrap(stack_hit[mask],baseline_hit[mask],val_gt[mask],
                                    n_boot=N_BOOTSTRAP,seed=SEED+11)
    records.append({'group':title,'queries':int(mask.sum()),'stacker_minus_large_pp':d,
                    'CI_low':lo,'CI_high':hi})
bootstrap=pd.DataFrame(records).set_index('group')
print('=== CLUSTERED BOOTSTRAP: Learned Stacker vs BLIP Large ===')
display(bootstrap.round(3))
bootstrap.to_csv(OUT/'stacker_vs_large_bootstrap.csv')

fig,axs=plt.subplots(1,2,figsize=(13,4.5))
results[['R@1','R@5','R@10']].plot(kind='bar',ax=axs[0],rot=35)
axs[0].set_title('Flickr8k Validation: Retrieval Recall')
axs[0].set_ylabel('Recall (%)');axs[0].grid(axis='y',alpha=.25)
results[['short_R1','medium_R1','long_R1']].plot(kind='bar',ax=axs[1],rot=35)
axs[1].set_title('Text-to-Image Recall@1 by Caption Length')
axs[1].set_ylabel('Recall@1 (%)');axs[1].grid(axis='y',alpha=.25)
plt.tight_layout()
plt.savefig(OUT/'validation_figures.png',dpi=180,bbox_inches='tight')
plt.show()

per_query=pd.DataFrame({
    'caption':val_data['texts'],
    'ground_truth_image':[val_data['names'][i] for i in val_gt],
    'caption_length':length,
    'GT_in_Top30':val_y>=0,
})
for name,ranked in ranks.items():
    safe=name.lower().replace(' ','_').replace('-','_')
    per_query[safe+'_top1']=[val_data['names'][i] for i in ranked[:,0]]
    per_query[safe+'_correct']=(ranked[:,0]==val_gt)
per_query.to_csv(OUT/'validation_per_query.csv',index=False)

## 11 · Fit the final stacker using validation labels and export it

After obtaining image-disjoint OOF diagnostics, train one final 16-parameter stacker on all available positive validation candidate lists for a number of epochs based on the **median inner-fold early-stopping epoch**. This is a distinct fitted model, which may behave differently from the OOF ensemble. We **do not** use test data to fit or select its weights.

In [ ]:
final_epochs=max(1,int(np.median(cv_table['best_inner_epoch'])))
FINAL_W=fit_stacker_fixed_epochs(val_X,val_y,final_epochs)
assert FINAL_W.shape==(16,) and np.isfinite(FINAL_W).all()
np.savez(OUT/'final_stacker_weights.npz',
         weights=FINAL_W,features=np.array(feature_names),
         alpha=np.array([WISE_ALPHA],dtype=np.float32),
         top_k=np.array([TOP_K],dtype=np.int32),
         fitted_epochs=np.array([final_epochs],dtype=np.int32))
print('Final stacker training epochs:',final_epochs)
print('Final 16 weights:')
display(pd.DataFrame({'feature':feature_names,'weight':FINAL_W}).round(4))

config={'seed':SEED,'clip_id':CLIP_ID,'rerankers':RERANKERS,
        'alpha':WISE_ALPHA,'top_k':TOP_K,'split':'random-image-level-75_12.5_12.5',
        'cv_folds':CV_FOLDS,'stacker_features':feature_names,
        'final_stacker_epochs':final_epochs,'wise_source':wise_source,
        'pretraining_overlap_verified':False,
        'test_previously_inspected_in_project':True}
(OUT/'run_config.json').write_text(json.dumps(config,indent=2))

## 12 · Optional final test-set evaluation (no further tuning)

**Test results should be read with the project's known caveat:** the same test split has been examined in earlier iterations, so this is **not a pristine holdout study**. There is **no optimization of alpha, fold settings or stacker weights on test labels**. This cell may take another 20–30 minutes on a T4.

In [ ]:
if not RUN_FINAL_TEST:
    print('Final test disabled. Set RUN_FINAL_TEST=True in Configuration and rerun required cells.')
else:
    assert test_cache is not None
    test_data=make_retrieval(test_cache,'test',select_image_count=None)
    test_scores=frozen_scores(test_data)
    test_X,test_feature_names=make_stacker_features(test_scores,test_data['texts'])
    assert feature_names==test_feature_names
    test_gt=np.asarray(test_data['gt'],dtype=np.int64)
    test_candidates=test_data['candidates']
    test_stacker_scores=test_X@FINAL_W
    test_rankings={
        'WiSE-FT Top-30':test_candidates,
        'BLIP Base':rank_candidates(test_candidates,test_scores['base']),
        'BLIP Large':rank_candidates(test_candidates,test_scores['large']),
        'BLIP-2 ITM':rank_candidates(test_candidates,test_scores['blip2']),
        'Learned Stacker (final-fit)':rank_candidates(test_candidates,test_stacker_scores),
    }
    test_results=pd.DataFrame([
        calc_metrics(name,ranked,test_gt,test_data['texts'])
        for name,ranked in test_rankings.items()
    ]).set_index('Model')
    print('=== TEST EVALUATION (previously inspected test split) ===')
    display(test_results.round(3))
    test_results.to_csv(OUT/'test_results.csv')
    # Paired cluster bootstrap for the pre-specified comparison to strong Large baseline
    a=test_rankings['Learned Stacker (final-fit)'][:,0]==test_gt
    b=test_rankings['BLIP Large'][:,0]==test_gt
    delta,lo,hi=paired_image_bootstrap(a,b,test_gt,n_boot=N_BOOTSTRAP,seed=SEED+222)
    print(f'Stacker vs BLIP Large: {delta:+.3f} pp; 95% cluster-bootstrap CI [{lo:+.3f},{hi:+.3f}] pp')
    pd.DataFrame([{'comparator':'BLIP Large','delta_pp':delta,'ci_low':lo,'ci_high':hi}]).to_csv(
        OUT/'test_stacker_vs_large_bootstrap.csv',index=False)
    per_test=pd.DataFrame({
        'query':test_data['texts'],
        'ground_truth_image':[test_data['names'][i] for i in test_gt],
        'BLIP_Large_top1':[test_data['names'][i] for i in test_rankings['BLIP Large'][:,0]],
        'Stacker_top1':[test_data['names'][i] for i in test_rankings['Learned Stacker (final-fit)'][:,0]],
        'BLIP_Large_correct':b,'Stacker_correct':a,
    })
    per_test.to_csv(OUT/'test_per_query.csv',index=False)

## 13 · Qualitative retrieval visualization and saved artifacts

In [ ]:
# Show a few validation queries and the learned OOF top-5 images.
rng=np.random.default_rng(SEED+313)
query_examples=rng.choice(len(val_gt),size=min(3,len(val_gt)),replace=False)
for qi in query_examples:
    print('\nQuery:',val_data['texts'][int(qi)])
    print('Ground-truth image:',val_data['names'][int(val_gt[qi])])
    row_ids=ranks['Learned Stacker OOF'][int(qi),:5]
    fig,axes=plt.subplots(1,5,figsize=(15,3.4))
    for position,(ax,image_id) in enumerate(zip(axes,row_ids),1):
        image_name=val_data['names'][int(image_id)]
        with Image.open(IMAGE_DIR/image_name) as img:
            ax.imshow(img.convert('RGB'))
        hit=image_id==val_gt[qi]
        ax.set_title(f'Rank {position}' + ('  [GT]' if hit else ''),fontsize=10)
        ax.axis('off')
    plt.tight_layout();plt.show()

print('\n=== OUTPUT FILES ===')
for file in sorted(OUT.iterdir()):
    if file.is_file() and file.suffix in ('.csv','.pt','.npz','.json','.png'):
        print(f'{file.name:48s} {file.stat().st_size/(1024**2):7.2f} MB')
print('\nFINAL PIPELINE: WiSE-FT CLIP Top-30 -> BLIP Base/Large/BLIP-2 ITM -> 16-feature learned stacker.')
print('No BLIP LoRA fine-tuning; no Qwen MLLM gate; no SigLIP union.')

## 14 · Optional real-world query: enter any caption and retrieve images

In [ ]:
# An end-to-end demo using a caption NOT necessarily present in Flickr8k captions.
# The gallery here is the validation image split; no labels are used to score this query.
RUN_FREE_TEXT_DEMO = False   # True to run: loads all three reranker checkpoints again
QUERY_TEXT = 'A brown dog jumping over a fence.'

@torch.inference_mode()
def text_to_image_search(text, gallery_cache, final_weights, top_k=TOP_K):
    # Stage 1. CLIP text embedding + WiSE-FT candidate retrieval.
    base=CLIPModel.from_pretrained(CLIP_ID).to(DEVICE).eval()
    tokenized=clip_processor.tokenizer(
        [text],padding='max_length',truncation=True,max_length=77,return_tensors='pt'
    )
    th=base.text_model(input_ids=tokenized['input_ids'].to(DEVICE),
                       attention_mask=tokenized['attention_mask'].to(DEVICE)).pooler_output.float()
    query=F.normalize(F.linear(th,WT),dim=-1)
    gallery=F.normalize(F.linear(gallery_cache['v'].to(DEVICE).float(),WV),dim=-1)
    similarity=(query @ gallery.T).squeeze(0)
    values,inds=similarity.topk(min(top_k,len(gallery)))
    del base,tokenized,th,query,gallery
    gc.collect();torch.cuda.empty_cache()

    # Stage 2. Frozen BLIP/Base/Large/BLIP2 matching scores over Top-30 only.
    demo_data={
        'split':'demo_'+sha(text),
        'names':gallery_cache['names'],
        'texts':[text],
        'candidates':inds.cpu().numpy()[None,:],
        'clip_scores':values.float().cpu().numpy()[None,:]
    }
    demo_scores=frozen_scores(demo_data)
    demo_x,_=make_stacker_features(demo_scores,[text])

    # Stage 3. Final learned listwise stacker.
    ranking=rank_candidates(demo_data['candidates'],demo_x@final_weights)[0]
    return [(demo_data['names'][int(i)],int(i)) for i in ranking[:5]]

if RUN_FREE_TEXT_DEMO:
    retrieved=text_to_image_search(QUERY_TEXT,val_cache,FINAL_W)
    print('Query:',QUERY_TEXT)
    fig,axes=plt.subplots(1,len(retrieved),figsize=(16,3.4))
    for rank,(filename,image_id) in enumerate(retrieved,1):
        with Image.open(IMAGE_DIR/filename) as img:
            axes[rank-1].imshow(img.convert('RGB'))
        axes[rank-1].set_title(f'Rank {rank}')
        axes[rank-1].axis('off')
    plt.tight_layout();plt.show()
else:
    print('Free-text demo is disabled. Set RUN_FREE_TEXT_DEMO=True to try a custom caption.')

## Interpretation and reporting checklist

1. **Primary performance**: report full-validation **image-disjoint OOF** results, the **BLIP Large-only** baseline, and the **cluster-bootstrap CI** for the learned stacker vs BLIP Large.
2. **Length-specific effects**: show short (≤7 words), medium (8–13), long (≥14) Recall@1; never infer causation from caption length alone.
3. **Ceiling**: Top-30 candidate coverage bounds any pure reranker; final Recall@10 is ≤ ceiling.
4. **Final test**: distinguish the **single final-fitted** stacker from the OOF models; previous test usage reduces independence.
5. **Limitations**: possible valid alternative images, custom Flickr8k splits, incomplete verification of checkpoint pretraining overlap, GPU/runtime overhead and small CLIP adaptation dataset.
6. **Scope**: an effective engineering project and a useful research baseline. Model ensembling, WiSE-FT and listwise stacking are established techniques; **do not claim new algorithmic novelty** without a separate contribution.

All outputs live under `/kaggle/working/final_wise_blip_stacker/`. For final sharing, download notebook plus `wiseft_projections.pt`, `final_stacker_weights.npz`, `run_config.json`, and results CSVs.